[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/ruppeiner/ruppeiner.ipynb)

# The Ruppeiner Metric Near the Critical Point

This example computes the Ruppeiner fluctuation metric and scalar curvature of a van der Waals fluid near its critical point. Thermal agitation continually perturbs a fluid's temperature and volume around equilibrium, and the fluctuation metric measures the thermodynamic cost that suppresses large excursions. Near the critical point, pressure barely changes with volume, so volume fluctuations grow cheap, equal-cost ellipses stretch horizontally, and the scalar curvature of the metric diverges.

Two degenerate directions `t0 v0` form an exterior state space in which a state is modelled as an antivector combining `t.dual()` and `v.dual()`, read back without a metric by `t & state` and `v & state`. The fluctuation metric is a symmetric two-dyad map `Potential <- State` whose pairing `metric(state)(variation) & variation` gives the cost of a state variation. Maxwell's relation cancels the cross term between temperature and volume variations, and each state's equal-cost contour is rendered directly as a projective conic.

In Riemannian tensor-index and thermodynamic partial-derivative notation, as in [Ruppeiner, 2010](https://arxiv.org/abs/1007.2160), the entropy Hessian $g_{ij} = -\frac{\partial^2 S}{\partial X^i \partial X^j}$ on $X = (U, V)$ pulls back to $(T, V)$ via $J^{\mathsf T} g J$ and $(\partial S/\partial V)_T = (\partial p/\partial T)_V$ as $ds^2 = \frac{C_V}{T^2}\,dT^2 - \frac{1}{T}\left(\frac{\partial p}{\partial V}\right)_T dV^2$, while contracting $R = g^{ik} g^{jl} R_{ijkl}$ from $\Gamma^k_{ij} = \tfrac12 g^{kl}(\partial_i g_{jl} + \partial_j g_{il} - \partial_l g_{ij})$ reduces on orthogonal coordinates to Brioschi's second-derivative ratio.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import show_states
from examples.thermodynamics.ruppeiner import render

np.set_printoptions(precision=4, suppress=True)

ga = Algebra("t0v0")
mv = NumpyContext(ga, np.float64).multivector
Scalar = ga.gatype.scalar()                                   # Scalar+
State = ga.gatype.antivector()                                # Antivector-
Potential = ga.gatype.vector()                                # Vector+
Metric = ga.gatype((Potential, State))                        # Potential+ <- State-

# Regressive pairings read temperature and volume without assuming a metric.
temperature = mv.t & State                                    # [] Scalar+ <- State-
volume = mv.v & State                                         # [] Scalar+ <- State-

# Reduced units measure temperature and volume in their critical values, energy in the gas constant times
# the critical temperature, and pressure in that energy per critical volume, giving critical pressure 3/8.
attraction, exclusion, capacity = 9 / 8, 1 / 3, 1.5

## Fluctuation metric

Expanding the availability around equilibrium pairs small changes in entropy and pressure with changes in temperature and volume. Maxwell's relation equates the cross derivatives of entropy with respect to volume and pressure with respect to temperature, so the cross terms cancel. The remaining thermal cost comes from the heat capacity divided by the squared temperature, and the mechanical cost comes from minus the isothermal volume slope of pressure divided by the temperature. Above the critical point, both entries are positive.

In [ ]:
def pressure(state: State) -> Scalar:
    """The van der Waals equation of state balances thermal pressure in the free volume against molecular attraction."""
    return temperature(state) / (volume(state) - exclusion) - attraction / volume(state).squared()


def metric(state: State) -> Metric:
    """The Ruppeiner metric maps each state variation to the potential that reads its fluctuation cost."""
    t, v = temperature(state), volume(state)                  # [...] Scalar+, Scalar+
    # Maxwell's relation cancels the cross term between temperature and volume fluctuations.
    thermal = capacity / t.squared()                          # [...] Scalar+
    mechanical = 1 / (v - exclusion).squared() - 2 * attraction / (t * v ** 3)   # [...] Scalar+
    return mv.t * thermal * temperature + mv.v * mechanical * volume             # [...] Potential+ <- State-


warm_state = mv.t.dual() * 1.8 + mv.v.dual()                  # [] State-
print("cost of a unit temperature variation:", metric(warm_state)(mv.t.dual()) & mv.t.dual())
print("cost of a unit volume variation:     ", metric(warm_state)(mv.v.dual()) & mv.v.dual())

## Scalar curvature across the phase plane

Scalar curvature measures how the fluctuation metric departs from a flat Euclidean plane, vanishing identically for an ideal gas without molecular attraction or excluded volume. Because the metric is diagonal in temperature and volume and its thermal entry depends on temperature alone, its scalar curvature reduces to rational expressions in the temperature derivatives of the mechanical entry and of the squared area element.

In [ ]:
def thermodynamic_curvature(state: State) -> Scalar:
    """Scalar curvature of the fluctuation metric equals twice its Gaussian curvature, which is positive on a sphere."""
    t, v = temperature(state), volume(state)                  # [...] Scalar+, Scalar+
    thermal = capacity / t.squared()                          # [...] Scalar+
    mechanical = 1 / (v - exclusion).squared() - 2 * attraction / (t * v ** 3)   # [...] Scalar+
    area_squared = thermal * mechanical                       # [...] Scalar+
    slope = 2 * attraction / (t.squared() * v ** 3)                              # [...] Scalar+
    slope_change = -4 * attraction / (t ** 3 * v ** 3)                           # [...] Scalar+
    area_change = -2 * thermal / t * mechanical + thermal * slope                # [...] Scalar+
    return -slope_change / area_squared + slope * area_change / (2 * area_squared.squared())


render.draw_landscape(metric, thermodynamic_curvature, pressure, warm_state);

Below the black spinodal curve on the temperature–volume plane, the mechanical entry turns negative and the uniform fluid is unstable, shaded grey. Above the spinodal, teal ellipses mark variations of equal cost around a lattice of stable states, while grey-blue contours of constant scalar curvature crowd toward the critical point at the spinodal peak. The warm state from above is highlighted in orange.

## Cooling toward the critical point

Cooling the fluid along the critical volume from 1.8 to 1.02 times the critical temperature drives the mechanical entry toward zero while leaving the thermal entry bounded. This cell and the next take about ten seconds, most of it drawing the animation.

In [ ]:
n_frames, frame_duration_ms = 96, 40
temperatures = np.linspace(1.8, 1.02, n_frames)

In [ ]:
def cooling_path(temperatures: np.ndarray) -> Iterator[State]:
    """Generates states along the critical volume cooling steadily through the given temperatures."""
    for frame in range(len(temperatures)):
        yield mv.t.dual() * temperatures[frame] + mv.v.dual()


cooling_states = show_states(
    cooling_path(temperatures), render.fluctuations, metric, thermodynamic_curvature, pressure,
    name="ruppeiner_fluctuations", duration_ms=frame_duration_ms, n_frames=n_frames,
)

As the orange state approaches the critical point on the left, its equal-cost ellipse stretches horizontally along volume while barely changing height along temperature; on the right, the orange tangent to its isotherm flattens toward zero slope.

## Critical scaling of the scalar curvature

Sampling states along the critical volume at logarithmically spaced offsets above the critical temperature reveals the power-law divergence of the scalar curvature.

In [ ]:
n_samples = 100
offsets = np.geomspace(0.002, 0.8, n_samples)
supercritical_states = mv.t.dual() * (1 + offsets) + mv.v.dual()              # [n_samples] State-
render.draw_scaling(supercritical_states, thermodynamic_curvature);

On logarithmic axes, the blue scalar-curvature curve runs parallel to the dashed grey reference line of slope -2, confirming inverse-square divergence above the critical temperature.

In [ ]:
# checks
path_states = stack(cooling_states)                                           # [n_frames] State-
# Along the cooling path, the thermal entry equals capacity over squared temperature and the cross term vanishes.
assert (((metric(path_states)(mv.t.dual()) & mv.t.dual()) - capacity / temperatures ** 2).abs() < 1e-12).all()
assert ((metric(path_states)(mv.t.dual()) & mv.v.dual()).abs() < 1e-12).all()
# Near the critical point, scalar curvature approaches one third of the inverse square of the temperature offset.
assert (thermodynamic_curvature(supercritical_states[0]) * (3 * offsets[0] ** 2) - 1).abs() < 0.01